In [10]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Acs1112@mysql",
    database="behaviorpulse"
)


In [3]:
import pandas as pd

df = pd.read_sql("SELECT * FROM activities", conn)
df.head()


C:\Users\chira\AppData\Local\Temp\ipykernel_15428\3720352896.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM activities", conn)


,record_id,user_id,activity_date,day_of_week,activity_type,planned_time_hrs,actual_time_hrs,completion_status,interruption_count,energy_level,mood,invalid_time_flag,energy_missing_flag,duplicate_flag,efficiency,burnout_risk,high_interruption_flag
0,1,101,2025-11-01,mon,study,3.0,2.5,yes,2,4,high,0,0,0,0.833333,0,0
1,2,101,2025-11-02,tue,study,2.0,3.0,yes,5,3,neutral,0,0,0,1.500000,0,1
2,3,101,2025-11-03,wed,study,3.0,1.5,no,7,3,low,0,1,0,0.500000,0,1
3,4,101,2025-11-04,thu,work,4.0,4.5,yes,1,5,high,0,0,0,1.125000,0,0
4,5,101,2025-11-05,fri,rest,1.0,0.5,no,3,2,low,0,0,0,0.500000,0,0


In [4]:
df.describe()


,record_id,user_id,planned_time_hrs,actual_time_hrs,interruption_count,energy_level,invalid_time_flag,energy_missing_flag,duplicate_flag,efficiency,burnout_risk,high_interruption_flag
count,120.000000,120.000000,120.000000,115.000000,120.000000,120.000000,120.000000,120.000000,120.000000,115.000000,120.0,120.000000
mean,15.500000,101.466667,2.862500,2.660870,3.066667,3.525000,0.041667,0.200000,0.375000,0.933396,0.0,0.300000
std,8.691733,0.500979,1.210879,1.614827,2.539547,0.995473,0.200664,0.401677,0.486153,0.512752,0.0,0.460179
min,1.000000,101.000000,0.500000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
25%,8.000000,101.000000,2.000000,1.500000,1.000000,3.000000,0.000000,0.000000,0.000000,0.666667,0.0,0.000000
50%,15.500000,101.000000,3.000000,2.500000,2.500000,4.000000,0.000000,0.000000,0.000000,0.888889,0.0,0.000000
75%,23.000000,102.000000,3.500000,3.500000,5.000000,4.000000,0.000000,0.000000,1.000000,1.154765,0.0,1.000000
max,30.000000,102.000000,6.000000,7.500000,9.000000,5.000000,1.000000,1.000000,1.000000,3.000000,0.0,1.000000


In [5]:
df.groupby('day_of_week')['burnout_risk'].mean()


day_of_week
fri    0.0
mon    0.0
sat    0.0
sun    0.0
thu    0.0
tue    0.0
wed    0.0
Name: burnout_risk, dtype: float64

In [6]:
df.groupby('activity_type')['burnout_risk'].mean()


activity_type
rest     0.0
study    0.0
work     0.0
Name: burnout_risk, dtype: float64

In [7]:
df['burnout_score'] = (
    (df['actual_time_hrs'] - df['planned_time_hrs']).clip(lower=0)
    + df['interruption_count'] * 0.5
    + (5 - df['energy_level'])
)


In [8]:
df.groupby('day_of_week')['burnout_score'].mean()
df.groupby('activity_type')['burnout_score'].mean()


activity_type
rest     3.289474
study    2.801471
work     3.821429
Name: burnout_score, dtype: float64

In [13]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Acs1112@mysql",
    database="behaviorpulse"
)

cursor = conn.cursor()


In [ ]:
cursor.execute("""
ALTER TABLE activities
ADD COLUMN burnout_score FLOAT
""")
conn.commit()


In [ ]:
update_query = """
UPDATE activities
SET burnout_score = %s
WHERE record_id = %s
"""

data = list(zip(df['burnout_score'], df['record_id']))
cursor.executemany(update_query, data)
conn.commit()


In [ ]:
df_check = pd.read_sql(
    "SELECT record_id, burnout_score FROM activities LIMIT 5",
    conn
)
df_check
